In [1]:
# this notebook will scale the data and
# train test split
import pandas as pd
import json
from pathlib import Path

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

folder_path = find_project_root() / "data" / "mlData"

model_name = "BTCUSD-5m"
src_path = folder_path / "clean" / f"{model_name}-v5-A-input.jsonl"

# Read JSONL file - keep timestamp as raw number
df = pd.read_json(src_path, lines=True, convert_dates=False)
df.head()

,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,hl_pct
0,1704066000000,42241.089844,60.923592,67.243118,1704066000000,0,0,0.000237,0.696150,0.000237,0.001592,0.000710
1,1704066300000,42269.828125,84.640320,66.823570,1704066300000,1,1,0.000680,0.963925,0.000680,0.001581,0.001013
2,1704066600000,42222.101562,76.113548,67.325706,1704066600000,1,1,-0.001129,0.863871,-0.001129,0.001595,0.001151
3,1704066900000,42283.578125,81.403389,66.089752,1704066900000,1,1,0.001456,0.919758,0.001456,0.001563,0.001454
4,1704067200000,42397.230469,155.257309,67.278091,1704067200000,1,1,0.002688,1.737683,0.002688,0.001587,0.003213


In [2]:
# df = df.drop(columns=["timestamp"])
print(df.shape)
print(df.columns)
df.head()

(228677, 12)
Index(['timestamp', 'close', 'vol', 'atr42', 'ts_5m', 'label', 'train_mask',
       'body_pct', 'vol_norm', 'close_ret1', 'atr42_pct', 'hl_pct'],
      dtype='object')


,timestamp,close,vol,atr42,ts_5m,label,train_mask,body_pct,vol_norm,close_ret1,atr42_pct,hl_pct
0,1704066000000,42241.089844,60.923592,67.243118,1704066000000,0,0,0.000237,0.696150,0.000237,0.001592,0.000710
1,1704066300000,42269.828125,84.640320,66.823570,1704066300000,1,1,0.000680,0.963925,0.000680,0.001581,0.001013
2,1704066600000,42222.101562,76.113548,67.325706,1704066600000,1,1,-0.001129,0.863871,-0.001129,0.001595,0.001151
3,1704066900000,42283.578125,81.403389,66.089752,1704066900000,1,1,0.001456,0.919758,0.001456,0.001563,0.001454
4,1704067200000,42397.230469,155.257309,67.278091,1704067200000,1,1,0.002688,1.737683,0.002688,0.001587,0.003213


# Prepare input and output
```
Label balance (trainable only):
  Up   ( 1) : 76,205  (49.5%)
  Down (-1) : 77,788  (50.5%)
  Total     : 153,993  (67.3% of clean rows)
  Timeout   : 74,684
```

In [3]:
# Sort chronologically first — never shuffle
df = df.sort_values('timestamp').reset_index(drop=True)
n = len(df)

# Chronological split on ALL rows (includes timeout rows — preserves time boundaries)
train_all = df.iloc[:int(n*0.8)]
val_all   = df.iloc[int(n*0.8):int(n*0.9)]
test_all  = df.iloc[int(n*0.9):]

# Apply train_mask: filter to trainable rows (label 1 or -1) within each split
train_df = train_all[train_all['train_mask'] == 1].copy()
val_df   = val_all[val_all['train_mask'] == 1].copy()
test_df  = test_all[test_all['train_mask'] == 1].copy()

print(f"All rows  — Train: {len(train_all):,} | Val: {len(val_all):,} | Test: {len(test_all):,}")
print(f"Trainable — Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")


All rows  — Train: 182,941 | Val: 22,868 | Test: 22,868
Trainable — Train: 123,609 | Val: 15,340 | Test: 15,044


# However — there's a catch for your case. You're using a chronological split, pytorch can't do

In [4]:
# apply scaler
from sklearn.preprocessing import StandardScaler
import joblib
from datetime import date

formatted_date = date.today().strftime("%Y%m")

# Drop redundant timestamp alias
train_df = train_df.drop(columns=["ts_5m"])
val_df   = val_df.drop(columns=["ts_5m"])
test_df  = test_df.drop(columns=["ts_5m"])

# Scale all features except non-numeric / target columns
scale_cols = ["close", "vol", "atr42", "body_pct", "vol_norm", "close_ret1", "atr42_pct", "hl_pct"]

scaler = StandardScaler()
train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])  # fit + transform on train only
val_df[scale_cols]   = scaler.transform(val_df[scale_cols])        # transform only
test_df[scale_cols]  = scaler.transform(test_df[scale_cols])       # transform only

joblib.dump(scaler, folder_path / "scaler" / f"{formatted_date}-{model_name}-scaler-A.pkl")
print("Scaler saved.")
print(f"Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")


Scaler saved.
Train: (123609, 11) | Val: (15340, 11) | Test: (15044, 11)


In [5]:
# save train data
# where Y = 0 around 26 cells, remove them during training
train_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-train-A.jsonl"
val_path = folder_path / "trainData" / f"{formatted_date}-{model_name}-val-A.jsonl"
test_path  = folder_path / "trainData" / f"{formatted_date}-{model_name}-test-A.jsonl"

train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_json(train_path, orient="records", lines=True)
val_df.to_json(val_path, orient="records", lines=True)
test_df.to_json(test_path,  orient="records", lines=True)

print(f"Saved train: {train_path.name}")
print(f"Saved val: {val_path.name}")
print(f"Saved test:  {test_path.name}")


Saved train: 202603-BTCUSD-5m-train-A.jsonl
Saved val: 202603-BTCUSD-5m-val-A.jsonl
Saved test:  202603-BTCUSD-5m-test-A.jsonl
